# Forecasting Demo

## Imports and Setup

In [9]:
from pathlib import Path
import sys

project_root = Path.cwd().parent.parent
sdk_path = project_root / "src" / "sdk" / "python"
sdk_path = sdk_path.resolve()

sys.path.insert(0, str(sdk_path))

In [10]:
MODEL_PATH = Path().cwd() / "AutogluonModels" / "ag-20260125_143848"
TEST_DATA_DIR = Path().cwd() / "data" / "scada_test.parquet"

## Load data and model
We prepared data and model in advance to this demo. Here we are loading the prepared model and test data.

In [12]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.master("local[*]")
    .appName("SCADA-Forecasting")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "8g")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.sql.shuffle.partitions", "50")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate()
)

# Load test data
test_data_all = spark.read.parquet(str(TEST_DATA_DIR))

# Filter to only one turbine for faster demo
TURBINE_ID = "5_Kelmarsh"  # Focus on single turbine
test_data = test_data_all.filter(test_data_all.item_id == TURBINE_ID)

print(f"Loaded data for turbine: {TURBINE_ID}")
print(f"Total rows: {test_data.count()}")
print(f"Time range: {test_data.agg({'timestamp': 'min'}).collect()[0][0]} to {test_data.agg({'timestamp': 'max'}).collect()[0][0]}")

Loaded data for turbine: 5_Kelmarsh
Total rows: 8784
Time range: 2020-01-01 00:00:00 to 2020-12-31 23:00:00


In [3]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.master("local[*]")
    .appName("SCADA-Forecasting")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "8g")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.sql.shuffle.partitions", "50")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate()
)

# load test data
test_data = spark.read.parquet(str(TEST_DATA_DIR))

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/26 11:25:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [13]:
from rtdip_sdk.pipelines.forecasting.spark.autogluon_timeseries import (
    AutoGluonTimeSeries,
)

# load AutoGluon model from disk
model = AutoGluonTimeSeries().load_model(MODEL_PATH)

Model loaded from /workspaces/amos2025ws03-rtdip-timeseries-forecasting/amos_team_resources/demoday/AutogluonModels/ag-20260125_143848


## Forecast data

Now that we loaded both data and model let's forecast our TS data!

We create a data split.

In [15]:
# Sort by timestamp
test_data_sorted = test_data.orderBy("timestamp")

total_rows = test_data_sorted.count()
split_point = int(total_rows * 0.67)

print(f"Data split for {TURBINE_ID}:")
print(f"  Total rows: {total_rows}")
print(f"  Split at row: {split_point}")

test_data_pd = test_data_sorted.toPandas()

# Split into historic and ground truth
historic_data_pd_split = test_data_pd.iloc[:split_point]
ground_truth_pd_split = test_data_pd.iloc[split_point:]


# Convert back to Spark DataFrames
historic_data = spark.createDataFrame(historic_data_pd_split)
ground_truth = spark.createDataFrame(ground_truth_pd_split)

Data split for 5_Kelmarsh:
  Total rows: 8784
  Split at row: 5885


In [16]:
# Generate predictions based on historic data
predictions = model.predict(historic_data)

Now we can visualize the forecasting results.

In [29]:
from rtdip_sdk.pipelines.visualization.plotly.forecasting import (
    ForecastComparisonPlotInteractive,
)

import pandas as pd

# Convert Spark DataFrames to Pandas for visualization (single turbine only)
historic_data_pd = historic_data.toPandas()
predictions_pd = predictions.toPandas()
actual_data_pd = ground_truth.toPandas()

# Extract timestamps and values
forecast_start = predictions_pd['timestamp'].min()
forecast_end = predictions_pd['timestamp'].max()

# Prepare historic data: last 24 hours
historic_start = forecast_start - pd.Timedelta(hours=24)
historic_viz_24h = historic_data_pd[historic_data_pd['timestamp'] >= historic_start][['timestamp', 'target']].copy()
historic_viz_24h.rename(columns={'target': 'value'}, inplace=True)

# Prepare forecast data
predictions_viz = predictions_pd[['timestamp', 'mean']].copy()

# Prepare actual data: match forecast period and close the gap
actual_filtered = actual_data_pd[
    (actual_data_pd['timestamp'] >= forecast_start) &
    (actual_data_pd['timestamp'] <= forecast_end)
][['timestamp', 'target']].copy()
actual_filtered.rename(columns={'target': 'value'}, inplace=True)

# Add last historic point to close the gap
last_historic = historic_viz_24h.iloc[[-1]].copy()
actual_viz_with_continuity = pd.concat([last_historic, actual_filtered], ignore_index=True)

# Create visualization
fig = ForecastComparisonPlotInteractive(
    historical_data=historic_viz_24h,
    forecast_data=predictions_viz,
    actual_data=actual_viz_with_continuity,
    forecast_start=forecast_start,
    sensor_id=TURBINE_ID,
    title="Power Forecast (24h)"
)

fig.plot().show()